In [2]:
## Baseline model

import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

def load_splits(path='dataset_splits.json'):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def evaluate_10fold_cv(make_pipeline_fn, cv_df, text_col, test_texts, y_test,
                      n_folds=10):
    """
    Ported from reference notebook. Fits TF-IDF inside each fold.
    """
    fold_accs = []
    fold_f1s = []

    for fold_idx in range(n_folds):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr = cv_df.loc[train_mask, text_col].tolist()
        y_tr = cv_df.loc[train_mask, "label_id"].values
        X_val = cv_df.loc[val_mask, text_col].tolist()
        y_val = cv_df.loc[val_mask, "label_id"].values

        pipe = make_pipeline_fn()
        pipe.fit(X_tr, y_tr)
        val_preds = pipe.predict(X_val)

        fold_accs.append(accuracy_score(y_val, val_preds))
        fold_f1s.append(f1_score(y_val, val_preds, average="macro"))

    # Final fit on all CV data to evaluate on test set
    final_pipe = make_pipeline_fn()
    X_cv_all = cv_df[text_col].tolist()
    y_cv_all = cv_df["label_id"].values
    final_pipe.fit(X_cv_all, y_cv_all)

    test_preds = final_pipe.predict(test_texts)
    test_acc = accuracy_score(y_test, test_preds)
    test_f1 = f1_score(y_test, test_preds, average="macro")

    return {
        "cv_acc_mean": np.mean(fold_accs),
        "cv_acc_std": np.std(fold_accs),
        "cv_f1_mean": np.mean(fold_f1s),
        "cv_f1_std": np.std(fold_f1s),
        "test_acc": test_acc,
        "test_f1": test_f1,
        "report": classification_report(y_test, test_preds, target_names=["no", "maybe", "yes"])
    }

def make_baseline_pipe():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, stop_words="english",
            ngram_range=(1, 2), max_features=30000
        )),
        ("lr", LogisticRegression(
            class_weight="balanced", random_state=42, max_iter=1000
        ))
    ])

def run_baseline():
    splits = load_splits()
    cv_df = pd.DataFrame(splits['cv'])
    test_df = pd.DataFrame(splits['test'])

    y_test = test_df["label_id"].values
    X_test_qctx = test_df["q_ctx"].tolist()

    print("============================================================")
    print("BASELINE: TF-IDF (1,2)-grams + Logistic Regression")
    print("Protocol: 10-Fold CV on 500 samples, Test on 500 samples")
    print("============================================================")

    results = evaluate_10fold_cv(
        make_baseline_pipe,
        cv_df,
        "q_ctx",
        X_test_qctx,
        y_test
    )

    print(f"10-Fold CV Accuracy : {results['cv_acc_mean']:.4f} ± {results['cv_acc_std']:.4f}")
    print(f"10-Fold CV Macro-F1 : {results['cv_f1_mean']:.4f} ± {results['cv_f1_std']:.4f}")
    print(f"Test Accuracy       : {results['test_acc']:.4f}")
    print(f"Test Macro-F1       : {results['test_f1']:.4f}")
    print("\nTest Classification Report:")
    print(results['report'])

if __name__ == "__main__":
    run_baseline()


BASELINE: TF-IDF (1,2)-grams + Logistic Regression
Protocol: 10-Fold CV on 500 samples, Test on 500 samples
10-Fold CV Accuracy : 0.5260 ± 0.0754
10-Fold CV Macro-F1 : 0.3549 ± 0.0569
Test Accuracy       : 0.5120
Test Macro-F1       : 0.3558

Test Classification Report:
              precision    recall  f1-score   support

          no       0.39      0.30      0.34       169
       maybe       0.23      0.05      0.09        55
         yes       0.57      0.73      0.64       276

    accuracy                           0.51       500
   macro avg       0.40      0.36      0.36       500
weighted avg       0.47      0.51      0.48       500



In [3]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report


RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Load data
def load_data():
    with open('dataset_splits.json', 'r', encoding='utf-8') as f:
        splits = json.load(f)

    cv_df = pd.DataFrame(splits['cv'])
    test_df = pd.DataFrame(splits['test'])

    dev_fold_mask = cv_df["fold"] == 0
    train_fold_mask = ~dev_fold_mask

    # uses q_ctx_ans
    train_texts = cv_df.loc[train_fold_mask, "text_q_ctx_ans"].tolist()
    train_labels = cv_df.loc[train_fold_mask, "label_id"].values.tolist()

    val_texts = cv_df.loc[dev_fold_mask, "text_q_ctx_ans"].tolist()
    val_labels = cv_df.loc[dev_fold_mask, "label_id"].values.tolist()

    test_texts = test_df["text_q_ctx_ans"].tolist()
    test_labels = test_df["label_id"].values.tolist()

    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels


# Dataset and weights
class QADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=384):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

def get_class_weights(labels):
    counts = Counter(labels)
    total = len(labels)
    n_classes = 3
    weights = [total / (n_classes * counts[i]) for i in range(n_classes)]
    return torch.tensor(weights, dtype=torch.float32).to(device)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

# Run training
def run_finetuning():
    MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
    MAX_LEN = 384
    BATCH_SIZE = 8
    EPOCHS = 8
    LEARNING_RATE = 2e-5

    tr_txt, tr_lbl, val_txt, val_lbl, ts_txt, ts_lbl = load_data()
    weights = get_class_weights(tr_lbl)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

    train_ds = QADataset(tr_txt, tr_lbl, tokenizer, MAX_LEN)
    val_ds = QADataset(val_txt, val_lbl, tokenizer, MAX_LEN)
    test_ds = QADataset(ts_txt, ts_lbl, tokenizer, MAX_LEN)

    steps_per_epoch = max(1, len(train_ds) // BATCH_SIZE)
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(0.1 * total_steps)

    training_args = TrainingArguments(
        output_dir="./biobert_output",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        report_to="none",
        fp16=True,
        save_total_limit=2
    )

    trainer = WeightedTrainer(
        class_weights=weights,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    print("\nStarting BioBERT Fine-tuning (Identical to Notebook)...")
    trainer.train()

    print("\nEvaluating on Test Set...")
    test_results = trainer.predict(test_ds)
    preds = np.argmax(test_results.predictions, axis=1)

    print("\nFinal Results:")
    print(f"Accuracy: {accuracy_score(ts_lbl, preds):.4f}")
    print(f"Macro F1: {f1_score(ts_lbl, preds, average='macro'):.4f}")
    print("\nClassification Report:")
    print(classification_report(ts_lbl, preds, target_names=["no", "maybe", "yes"]))

if __name__ == "__main__":
    run_finetuning()


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on Test Set...



Final Results:
Accuracy: 0.7120
Macro F1: 0.4930

Classification Report:
              precision    recall  f1-score   support

          no       0.70      0.66      0.68       169
       maybe       0.00      0.00      0.00        55
         yes       0.72      0.88      0.80       276

    accuracy                           0.71       500
   macro avg       0.48      0.52      0.49       500
weighted avg       0.64      0.71      0.67       500



In [4]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report


RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# load data
def load_data():
    with open('dataset_splits.json', 'r', encoding='utf-8') as f:
        splits = json.load(f)

    cv_df = pd.DataFrame(splits['cv'])
    test_df = pd.DataFrame(splits['test'])

    dev_fold_mask = cv_df["fold"] == 0
    train_fold_mask = ~dev_fold_mask

    # uses q_ctx
    train_texts = cv_df.loc[train_fold_mask, "q_ctx"].tolist()
    train_labels = cv_df.loc[train_fold_mask, "label_id"].values.tolist()

    val_texts = cv_df.loc[dev_fold_mask, "q_ctx"].tolist()
    val_labels = cv_df.loc[dev_fold_mask, "label_id"].values.tolist()

    test_texts = test_df["q_ctx"].tolist()
    test_labels = test_df["label_id"].values.tolist()

    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels

# Dataset and weights
class QADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=384):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

def get_class_weights(labels):
    counts = Counter(labels)
    total = len(labels)
    n_classes = 3
    weights = [total / (n_classes * counts[i]) for i in range(n_classes)]
    return torch.tensor(weights, dtype=torch.float32).to(device)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }


# Run training
def run_finetuning():
    MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
    MAX_LEN = 384
    BATCH_SIZE = 8
    EPOCHS = 8
    LEARNING_RATE = 2e-5

    tr_txt, tr_lbl, val_txt, val_lbl, ts_txt, ts_lbl = load_data()
    weights = get_class_weights(tr_lbl)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

    train_ds = QADataset(tr_txt, tr_lbl, tokenizer, MAX_LEN)
    val_ds = QADataset(val_txt, val_lbl, tokenizer, MAX_LEN)
    test_ds = QADataset(ts_txt, ts_lbl, tokenizer, MAX_LEN)

    steps_per_epoch = max(1, len(train_ds) // BATCH_SIZE)
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(0.1 * total_steps)

    training_args = TrainingArguments(
        output_dir="./biobert_output",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        report_to="none",
        fp16=True,
        save_total_limit=2
    )

    trainer = WeightedTrainer(
        class_weights=weights,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    print("\nStarting BioBERT Fine-tuning (Identical to Notebook)...")
    trainer.train()

    print("\nEvaluating on Test Set...")
    test_results = trainer.predict(test_ds)
    preds = np.argmax(test_results.predictions, axis=1)

    print("\nFinal Results:")
    print(f"Accuracy: {accuracy_score(ts_lbl, preds):.4f}")
    print(f"Macro F1: {f1_score(ts_lbl, preds, average='macro'):.4f}")
    print("\nClassification Report:")
    print(classification_report(ts_lbl, preds, target_names=["no", "maybe", "yes"]))

if __name__ == "__main__":
    run_finetuning()

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne


Starting BioBERT Fine-tuning (Identical to Notebook)...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.128297,1.103233,0.440000,0.312198
2,1.081091,1.083607,0.480000,0.259259
3,0.956305,1.097540,0.460000,0.356463
4,0.732548,1.218832,0.640000,0.427778
5,0.467090,1.348260,0.520000,0.366621
6,0.312480,1.375312,0.560000,0.401550


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on Test Set...



Final Results:
Accuracy: 0.6000
Macro F1: 0.4315

Classification Report:
              precision    recall  f1-score   support

          no       0.52      0.55      0.53       169
       maybe       0.20      0.04      0.06        55
         yes       0.66      0.74      0.70       276

    accuracy                           0.60       500
   macro avg       0.46      0.44      0.43       500
weighted avg       0.56      0.60      0.57       500

